[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tunnel-ai/way/blob/main/notebooks/05_aux_v2_pipeline.ipynb)

# Module 5, v2 Pipeline: Stratified Fetch + Embeddings + Log-Odds

**Notebook:** `05_aux_v2_pipeline`  
**Companion to:** `05_00_main`

## Why a v2?

The v1 notebook deliberately failed Act I to motivate Act II, then conditioned on AI governance and used **mean TF–IDF** to surface "characteristic" terms per region. That's a defensible *teaching* arc, but the modeling leaves real things on the table:

1. **The corpus was unbalanced.** Sorting OpenAlex by recency favors US/Europe-anchored venues, and v1's priority rule (US wins ties) further inflated the US share. v2 fetches each region **separately, to a quota**.
2. **The unit of analysis was sloppy.** v1 assigned a paper to a region if *any* affiliation matched. v2 uses **first-author affiliation** as the analytical unit.
3. **Mean TF–IDF is not distinctiveness.** v2 swaps it for **log-odds with an informative Dirichlet prior** (Monroe, Colaresi & Quinn, 2008, "Fightin' Words"), which actually answers the *comparative* question.
4. **TF–IDF treats synonyms as unrelated tokens.** v2 adds **sentence embeddings** for a semantic lens alongside the lexical one.

## What v2 still can't fix

- "AI governance" is itself an English-language, internationally-legible category. Papers that don't speak that vocabulary don't enter the corpus.
- First-author affiliation is still a **proxy** for "regional research agenda," not a measurement of one.
- 600 papers per region is small. Don't over-read any individual term.

Methodological rigor doesn't *manufacture* findings, it tells you which findings you're allowed to claim.

## 0) Setup

Same Colab-friendly setup as v1, plus `sentence-transformers` for the semantic lens.

In [ ]:
import os
import re
import time
import textwrap

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# sentence-transformers: ~80MB model, runs on CPU
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
    from sentence_transformers import SentenceTransformer

In [ ]:
DATA_DIR = "assets/data"
os.makedirs(DATA_DIR, exist_ok=True)
V2_CACHE_PATH = os.path.join(DATA_DIR, "openalex_v2_stratified.csv")

BASE = "https://api.openalex.org/works"
PER_REGION_QUOTA = 600
FROM_YEAR = 2022

## 1) Stratified fetch (one query per region)

We query OpenAlex once **per region** with an explicit `country_code` filter, until each region hits its quota. This is the single biggest fix relative to v1: it removes "whoever dominates the corpus dominates the answer."

OpenAlex filter conventions used here:
- `,` between filters means AND
- `|` between values inside one filter means OR (used for the European country list)

In [ ]:
REGION_QUERIES = {
    "United States": "US",
    "China":         "CN",
    "Europe":        "GB|DE|FR|NL|IT|ES|SE|CH|BE|DK|FI|NO|AT|IE|PT|PL|CZ|GR",
}

# Used later for re-deriving region from first-author affiliation
EUROPE_CODES = {
    "AT","BE","BG","HR","CY","CZ","DK","EE","FI","FR","DE","GR","HU","IE","IT",
    "LV","LT","LU","MT","NL","PL","PT","RO","SK","SI","ES","SE",
    "GB","UK","NO","CH","IS","LI",
}

In [ ]:
def inverted_index_to_text(inv):
    """Reconstruct text from OpenAlex inverted index. Same as v1."""
    if not isinstance(inv, dict) or len(inv) == 0:
        return None
    max_pos = 0
    for _, positions in inv.items():
        if positions:
            max_pos = max(max_pos, max(positions))
    tokens = [""] * (max_pos + 1)
    for token, positions in inv.items():
        for p in positions:
            if 0 <= p < len(tokens) and tokens[p] == "":
                tokens[p] = token
    text = " ".join(t for t in tokens if t)
    return text if text.strip() else None


def first_author_country(work):
    """Country code of the first author's primary institution. None if missing."""
    auths = work.get("authorships") or []
    # Prefer the explicit author_position field
    for auth in auths:
        if auth.get("author_position") == "first":
            for inst in (auth.get("institutions") or []):
                cc = inst.get("country_code")
                if cc:
                    return cc.upper()
            return None
    # Fallback: first item in authorships list
    if auths:
        for inst in (auths[0].get("institutions") or []):
            cc = inst.get("country_code")
            if cc:
                return cc.upper()
    return None


def all_country_codes(work):
    codes = []
    for auth in work.get("authorships") or []:
        for inst in (auth.get("institutions") or []):
            cc = inst.get("country_code")
            if cc:
                codes.append(cc.upper())
    return sorted(set(codes))

In [ ]:
def fetch_region(region_name, country_filter, quota):
    rows = []
    cursor = "*"
    fetched = 0
    while fetched < quota:
        params = {
            "per-page": 200,
            "cursor": cursor,
            "filter": (
                f"has_abstract:true,"
                f"from_publication_date:{FROM_YEAR}-01-01,"
                f"authorships.institutions.country_code:{country_filter}"
            ),
            "sort": "publication_date:desc",
        }
        r = requests.get(BASE, params=params, timeout=60)
        r.raise_for_status()
        payload = r.json()

        for work in (payload.get("results") or []):
            if fetched >= quota:
                break
            abstract = inverted_index_to_text(work.get("abstract_inverted_index"))
            if not abstract:
                continue
            rows.append({
                "openalex_id": work.get("id"),
                "title": work.get("title"),
                "publication_date": work.get("publication_date"),
                "first_author_country": first_author_country(work),
                "all_country_codes": "|".join(all_country_codes(work)),
                "abstract": abstract,
                "queried_region": region_name,
            })
            fetched += 1

        cursor = payload.get("meta", {}).get("next_cursor")
        if not cursor:
            break
        time.sleep(0.15)
    return pd.DataFrame(rows)

In [ ]:
if os.path.exists(V2_CACHE_PATH):
    df = pd.read_csv(V2_CACHE_PATH)
    print(f"Loaded cached: {len(df):,} rows from {V2_CACHE_PATH}")
else:
    parts = []
    for region, cc_filter in REGION_QUERIES.items():
        print(f"Fetching {region} (filter={cc_filter}) ...")
        part = fetch_region(region, cc_filter, PER_REGION_QUOTA)
        print(f"  -> {len(part)} papers")
        parts.append(part)
    df = pd.concat(parts, ignore_index=True)
    # Multi-region collaborations may appear in two queries, keep the first.
    df = df.drop_duplicates(subset="openalex_id").reset_index(drop=True)
    df.to_csv(V2_CACHE_PATH, index=False)
    print(f"\nSaved {len(df):,} rows to {V2_CACHE_PATH}")

df["queried_region"].value_counts()

## 2) Assign region by **first-author** affiliation

Important: `queried_region` tells us which OpenAlex query brought a paper in. That's not the same as "this paper's analytical region." We re-derive the analytical region from **first-author country** so multi-region collaborations don't get double-claimed.

Papers with no first-author country, or one outside our three buckets, are dropped. That's an explicit choice: we'd rather have three clean groups than four messy ones.

In [ ]:
def region_from_first_author(cc):
    if not isinstance(cc, str) or not cc:
        return "Unknown"
    if cc == "US":
        return "United States"
    if cc == "CN":
        return "China"
    if cc in EUROPE_CODES:
        return "Europe"
    return "Other"

df["region"] = df["first_author_country"].apply(region_from_first_author)

before = len(df)
df = df[df["region"].isin(["United States", "China", "Europe"])].reset_index(drop=True)
print(f"Kept {len(df):,} of {before:,} after restricting to US / China / Europe first authors.")
df["region"].value_counts()

## 3) Condition on AI governance

Same word-boundary regex as the fixed v1, applied to a **balanced** corpus this time. We also do the same minimal cleaning (lowercase, strip URLs, keep letters and hyphens, drop digits).

Heads-up: dropping digits is still an argument we're making. We keep it for direct comparability with v1.

In [ ]:
url_pat = re.compile(r"https?://\S+|www\.\S+")
multi_space_pat = re.compile(r"\s+")

def clean_text(s):
    if not isinstance(s, str):
        return ""
    s = url_pat.sub(" ", s.strip()).lower()
    s = re.sub(r"[^a-z\s\-]", " ", s)
    return multi_space_pat.sub(" ", s).strip()

df["text"] = df["abstract"].apply(clean_text)

GOV_TERMS = [
    r"governance", r"ethics", r"ethical", r"fairness",
    r"algorithmic\s+bias", r"accountability", r"transparency",
    r"privacy", r"regulation", r"regulatory", r"compliance",
    r"systemic\s+risk", r"responsible\s+ai", r"ai\s+safety",
    r"trustworthy\s+ai",
]
gov_pattern = r"\b(" + "|".join(GOV_TERMS) + r")\b"

df_gov = df[df["text"].str.contains(gov_pattern, regex=True)].reset_index(drop=True)

print(f"Balanced corpus:    {len(df):,}")
print(f"Governance subset:  {len(df_gov):,}")
df_gov["region"].value_counts()

## 4) Representation 1, counts (for log-odds)

For Monroe et al. log-odds, we want **raw counts**, not TF–IDF weights, the math expects integer occurrences. We keep the same `min_df` / `max_df` / `ngram_range` choices as v1 for direct comparability.

In [ ]:
count_vec = CountVectorizer(
    stop_words="english",
    min_df=5,
    max_df=0.9,
    ngram_range=(1, 2),
)
C = count_vec.fit_transform(df_gov["text"])
vocab = count_vec.get_feature_names_out()
print("Counts matrix:", C.shape)

### Log-odds with informative Dirichlet prior

For target group $A$ vs. comparison group $B$ over a shared vocabulary $V$:

- $y_w^{(i)}$ = count of word $w$ in group $i$
- $n^{(i)} = \sum_w y_w^{(i)}$ = total tokens in group $i$
- $\alpha_w$ = prior count for word $w$. We use **total corpus frequency** as the prior, it shrinks rare words toward the background distribution, which is exactly what we want.

Estimated log-odds for $w$ in group $i$:

$$\hat\Omega_w^{(i)} = \log\frac{y_w^{(i)} + \alpha_w}{n^{(i)} + \alpha_0 - y_w^{(i)} - \alpha_w}$$

Difference and (asymptotic) variance:

$$\hat\zeta_w^{(A-B)} = \hat\Omega_w^{(A)} - \hat\Omega_w^{(B)}, \qquad \sigma^2(\hat\zeta_w) \approx \frac{1}{y_w^{(A)} + \alpha_w} + \frac{1}{y_w^{(B)} + \alpha_w}$$

We rank by the z-score $\hat\zeta_w / \sigma$. Higher z = more distinctive of $A$ relative to $B$.

Why this beats mean TF–IDF for our question:
- It explicitly **compares** groups instead of describing one in isolation.
- It **shrinks** rare words toward zero rather than over-rewarding them (which TF–IDF can do via inverse document frequency).
- It returns a **z-score**, so we can talk about confidence, not just rank order.

Reference: Monroe, Colaresi & Quinn (2008), *Fightin' Words: Lexical Feature Selection and Evaluation for Identifying the Content of Political Conflict*.

In [ ]:
def log_odds_dirichlet(counts_a, counts_b, prior):
    """
    Returns z-scored log-odds difference (A vs. B) per word.
    All inputs are 1-D arrays over the same vocabulary.
    """
    counts_a = np.asarray(counts_a, dtype=float)
    counts_b = np.asarray(counts_b, dtype=float)
    prior    = np.asarray(prior,    dtype=float)

    n_a = counts_a.sum()
    n_b = counts_b.sum()
    a0  = prior.sum()

    # log-odds in each group, with smoothing
    log_p_a = np.log((counts_a + prior) / (n_a + a0 - counts_a - prior))
    log_p_b = np.log((counts_b + prior) / (n_b + a0 - counts_b - prior))

    delta = log_p_a - log_p_b
    var   = 1.0 / (counts_a + prior) + 1.0 / (counts_b + prior)
    return delta / np.sqrt(var)

In [ ]:
def region_counts(region):
    mask = (df_gov["region"] == region).values
    return np.asarray(C[mask].sum(axis=0)).ravel()

regions = ["United States", "Europe", "China"]
counts_by_region = {r: region_counts(r) for r in regions}

# Background prior: full-subset counts
prior = np.asarray(C.sum(axis=0)).ravel()

# Compare each region against the union of the other two
salience = {}
for r in regions:
    others = sum(counts_by_region[o] for o in regions if o != r)
    salience[r] = log_odds_dirichlet(counts_by_region[r], others, prior)

In [ ]:
TOP_N = 15

print("Top distinctive terms per region (z-scored log-odds vs. rest of governance subset)\n")
for r in regions:
    z = salience[r]
    top_idx = z.argsort()[::-1][:TOP_N]
    print(f"=== {r} ===")
    for i in top_idx:
        marker = "  ***" if z[i] >= 2.0 else ("  *" if z[i] >= 1.0 else "")
        print(f"  {vocab[i]:<28s}  z = {z[i]:+.2f}{marker}")
    print()

### How to read this output

These are words whose **odds of appearing** in this region's papers are most elevated relative to the other two regions. Because it's a comparative measure with a built-in variance term, the z-scores tell us how confident we should be:

| z-score | Interpretation |
| --- | --- |
| `≥ 2.0` (`***`) | Noticeably distinctive. Worth talking about. |
| `1.0–2.0` (`*`)  | Suggestive but not strong. Don't anchor a claim on it alone. |
| `< 1.0`         | Noise. You're seeing it because nothing stronger exists. |

**A region whose top list is dominated by z-scores under 2 is *not strongly differentiated* in this corpus.** That is itself a finding, and a more honest one than "here are the top mean-TF–IDF terms," which would have given you a list-shaped result regardless of whether any signal was actually there.

## 5) Representation 2, sentence embeddings (semantic view)

Counts and TF–IDF treat "regulation" and "governance" as unrelated tokens. Embeddings don't.

We encode each abstract with a small sentence-transformer (`all-MiniLM-L6-v2`, 384-dim, ~80 MB, CPU-friendly) and look at where each region's center of mass sits in semantic space.

In [ ]:
encoder = SentenceTransformer("all-MiniLM-L6-v2")
emb = encoder.encode(
    df_gov["abstract"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,  # so dot product == cosine
)
print("Embedding matrix:", emb.shape)

In [ ]:
centroids = {r: emb[(df_gov["region"] == r).values].mean(axis=0) for r in regions}

# Pairwise cosine similarity between regional centroids
n = len(regions)
sim_mat = np.zeros((n, n))
for i, ri in enumerate(regions):
    for j, rj in enumerate(regions):
        ci, cj = centroids[ri], centroids[rj]
        sim_mat[i, j] = float(np.dot(ci, cj) / (np.linalg.norm(ci) * np.linalg.norm(cj)))

sim_df = pd.DataFrame(sim_mat, index=regions, columns=regions).round(4)
print("Pairwise centroid cosine similarity:")
sim_df

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(sim_mat, cmap="viridis", vmin=sim_mat.min(), vmax=1.0)
ax.set_xticks(range(n)); ax.set_xticklabels(regions, rotation=20, ha="right")
ax.set_yticks(range(n)); ax.set_yticklabels(regions)
midpoint = (sim_mat.min() + sim_mat.max()) / 2
for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{sim_mat[i,j]:.3f}", ha="center", va="center",
                color="white" if sim_mat[i,j] < midpoint else "black", fontsize=10)
ax.set_title("Regional centroid cosine similarity\n(governance subset, sentence-embedding space)")
plt.tight_layout()
plt.show()

### What the centroid table says

Three things to check, in this order:

1. **All cosines very high (>0.95)?** Then regions are essentially overlapping in this space, the *topic* is dominating the signal. This is the most likely outcome on a corpus this small, and it's the honest one.
2. **One pair noticeably lower than the others?** That's where the largest semantic distance lives. Don't over-read it: it could be driven by a single subtopic over-represented in one region. Cross-check with the log-odds tables.
3. **Symmetry sanity check.** The matrix should be symmetric and have 1.0 on the diagonal. If not, something went wrong upstream.

## 6) Combining the two lenses

Lexical (log-odds) and semantic (embeddings) views can disagree. The pattern of (dis)agreement is itself diagnostic:

| Lexical signal | Semantic signal | Reading |
| --- | --- | --- |
| Strong | Strong | Most credible "regional emphasis difference." |
| Strong | Weak | Same concepts, different vocabularies (the synonym case). |
| Weak   | Strong | Same vocabulary, different conceptual neighborhoods. Rare; worth digging into. |
| Weak   | Weak   | The topic dominates; regional signal is below the noise floor in this sample. |

Your discussion should reference the table you actually got, not the table you were hoping for.

## 7) What this pipeline still can't tell you

A short, honest list, useful when someone reads your output and over-reads it:

- **OpenAlex coverage is uneven.** Chinese-language venues are under-represented in the index. We are still measuring "Chinese-affiliated papers that publish in OpenAlex-indexed (mostly English) venues", a subset of Chinese AI-governance writing, not the whole.
- **First-author affiliation ≠ funder ≠ research-program origin.** A US-affiliated postdoc trained in Beijing publishing in a UK journal is one paper with three plausible "regions."
- **600 papers per region is small.** Top-distinctive terms with z < 2 are not findings.
- **"AI governance" is a self-selecting category.** Papers that don't speak this vocabulary aren't in the corpus. We can only see the part of the discourse that already shares our shibboleths.
- **Emphasis ≠ agenda.** Emphasis is what shows up in abstracts. Agendas are what gets *funded*, *prioritized*, and *enacted*, and most of that is invisible from text alone.

The honest framing for a student: *we built a more careful pipeline, and that lets us say something more cautiously, about a smaller question, with explicit error bars.* That's the win. Methodological rigor doesn't manufacture findings, it tells you which findings you're allowed to claim.